# Interface Analysis — Tissue Boundary Characterization

This notebook demonstrates the unified interface analysis pipeline in spatioloji_s:

1. **Interface detection** — grid-based boundary between two tissue regions
2. **Region classification** — assign every cell to region A or B
3. **Gradient analysis** — how genes/features change across the boundary
4. **Infiltration scoring** — are immune cells penetrating into a region?

All functions work identically for point-based and polygon-based data — only cell centroid coordinates are used. No spatial graph required.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import spatioloji_s as sj

# Access interface functions from either module — they're the same
# sj.spatial.point.identify_interface is sj.spatial.polygon.identify_interface
import spatioloji_s.spatial.point as spoint

# Load your spatioloji object
# sp = sj.read_cosmx(...)  or  sj.read_xenium(...)

# Check available cell types
print(sp.cell_meta["cell_type"].value_counts())

---
## 1. Interface Detection (Grid Method)

Partitions the tissue into a spatial grid, majority-votes each tile as region A or B, and extracts contour lines where A-tiles border B-tiles. Produces clean boundaries that match what you'd draw by eye.

**No spatial graph needed** — only cell coordinates and group labels.

### Key parameters

| Parameter | Effect |
|-----------|--------|
| `grid_resolution` | Tiles along the longest axis. Higher = more detail, lower = smoother. Default 50. |
| `distance_threshold` | Max distance from contour to be labeled "interface". Auto-estimated if `None`. |
| `min_interface_cells` | Drop segments with fewer than N cells on either side. |
| `close_contours` | Close loops for enclosed regions (e.g. tumor islands in stroma). |

In [ ]:
# Identify interface between two regions
# Change region_a and region_b to match your cell_type labels
result = spoint.identify_interface(
    sp,
    group_col='cell_type',       # column in cell_meta with cell labels
    region_a='Tumor',            # one side of the boundary
    region_b='Stroma',           # other side
    grid_resolution=50,          # tile count along longest axis
    min_interface_cells=10,      # drop tiny segments
    coord_type='global',
    store=True,                  # saves 'interface_label' to cell_meta
)

print(f"\nSummary: {result.summary}")

In [ ]:
# Visualize — cells colored by interface role, contour lines overlay
sj.visualization.plot_interface_point_map(
    sp, result,
    color_a='#e74c3c',     # red for region A
    color_b='#2ecc71',     # green for region B
    point_size=0.5,
    contour_width=1.5,
    dpi=200,
)

### Tuning grid_resolution

Higher resolution captures finer detail but may produce noisy boundaries. Lower resolution gives smoother, coarser boundaries.

In [ ]:
resolutions = [20, 50, 100, 200]

fig, axes = plt.subplots(1, len(resolutions), figsize=(7 * len(resolutions), 8))

for ax, res in zip(axes, resolutions):
    r = spoint.identify_interface(
        sp,
        group_col='cell_type', region_a='Tumor', region_b='Stroma',
        grid_resolution=res, min_interface_cells=5,
        coord_type='global', store=False,
    )
    sj.visualization.plot_interface_point_map(
        sp, r, ax=ax, show=False, point_size=0.3,
        color_a='#e74c3c', color_b='#2ecc71',
        title=f"res={res} ({r.summary['n_segments']} segs)",
    )

plt.suptitle("Grid Resolution Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Per-segment metrics

Each contour segment has metrics: `length`, `tortuosity`, `n_cells_a`, `n_cells_b`.

**Tortuosity** = actual length / straight-line distance between endpoints.
- ~1.0 = smooth pushing border (better prognosis)
- \>1.5 = irregular/invasive front (worse prognosis)

In [ ]:
# Segment metrics table
print(result.segments[['segment_id', 'length', 'tortuosity', 'n_cells_a', 'n_cells_b']])

# Bar charts
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['length', 'tortuosity', 'n_cells_a']):
    sj.visualization.plot_interface_point_metrics(
        result, metric=metric, ax=ax, show=False,
    )
plt.tight_layout()
plt.show()

---
## 2. Region Classification

`classify_regions` assigns **every** cell (including immune, unclassified, etc.) to region A or B based on which side of the interface it falls on. Uses a high-resolution grid with majority voting.

The `min_segment_length` parameter filters out small noisy zones — zones with a perimeter shorter than this value are absorbed into the surrounding region.

In [ ]:
# Classify all cells as region_a or region_b
sides = spoint.classify_regions(
    sp, result,
    group_col='cell_type',
    min_segment_length=500,   # filter small noisy zones (perimeter in coord units)
    coord_type='global',
    store=True,               # saves 'region_side' to cell_meta
)

print(sides.value_counts())

---
## 3. Gradient Analysis

How do genes and cell features change as you move across the interface?

Positive distance = region A side, negative distance = region B side.

### 3a. Gene expression gradients

In [ ]:
# Compute gradient for selected genes
gradient = spoint.compute_gradient(
    sp,
    interface_result=result,
    genes=['EPCAM', 'VIM', 'COL1A1', 'MMP2', 'CD274', 'TGFB1'],  # adjust to your panel
    layer='log_normalized',      # use normalized expression
    n_bins=20,
    coord_type='global',
)

# Which genes change most across the boundary?
print("Gene gradients (sorted by |coefficient|):")
print(gradient.gene_gradients.sort_values('coef', key=abs, ascending=False))

# Plot gradient curves
sj.visualization.plot_gradient_curve(gradient)

### 3b. Cell meta feature gradients

Analyze how morphology metrics, pathway activity scores, or any numeric `cell_meta` column changes across the interface.

In [ ]:
# Gradient of cell_meta features
# Adjust feature names to what's available in your cell_meta
gradient_features = spoint.compute_gradient(
    sp,
    interface_result=result,
    genes=None,                  # skip genes
    features=[
        'morph_area',            # cell size
        'morph_circularity',     # cell shape roundness
        'morph_elongation',      # cell elongation
    ],
    n_bins=20,
    coord_type='global',
)

print("Feature gradients:")
print(gradient_features.gene_gradients)

sj.visualization.plot_gradient_curve(gradient_features)

### 3c. Cell filtering

Focus the gradient on specific subsets of cells.

In [ ]:
# Only region A + B cells (exclude immune / unclassified "other" cells)
gradient_regions = spoint.compute_gradient(
    sp, result, genes=['EPCAM', 'VIM'],
    cells='regions_only',
    layer='log_normalized',
)

# Only specific cell types
gradient_types = spoint.compute_gradient(
    sp, result, genes=['EPCAM', 'VIM'],
    cell_types=['Tumor', 'Stroma', 'Macrophage'],
    cell_type_col='cell_type',
    layer='log_normalized',
)

### 3d. Spatial distance field

Visualize the signed distance from the interface to verify the boundary makes biological sense.

In [ ]:
# Visualize signed distance from the interface
sj.visualization.plot_spatial_distance(
    sp, gradient.distances,
    interface_result=result,     # overlay contour lines
    show=True,
)

---
## 4. Infiltration Scoring

Are immune cells (CD8 T cells, macrophages, etc.) penetrating into the target region, or are they excluded at the boundary?

The function:
1. Computes signed distance of every cell to the interface
2. Auto-detects the target region (the side with *fewer* immune cells — they're the visitors)
3. Classifies each immune cell as "infiltrating" or "resident"
4. Fits a density gradient (slope + p-value) per immune type

In [ ]:
# Score immune infiltration
# Adjust immune_types to match your cell type labels
infiltration = spoint.score_infiltration(
    sp,
    interface_result=result,
    immune_col='cell_type',                              # column with cell labels
    immune_types=['CD8_T', 'Macrophage', 'B_cell'],      # immune cell labels
    target_region='Tumor',                               # or None for auto-detect
    depth_bins=10,
    coord_type='global',
)

# Per-type metrics
print(infiltration.per_type_metrics)

# Visualize
sj.visualization.plot_infiltration_summary(infiltration)

---
## 5. Auto-discover Gene Programs

Find gene modules that co-vary across the interface using NMF or PCA.

In [ ]:
# Auto-discover gene programs via NMF
gradient_programs = spoint.compute_gradient(
    sp, result,
    genes=None,                  # use all genes
    layer='log_normalized',
    auto_programs='nmf',         # or 'pca'
    n_auto_programs=5,
    n_bins=20,
)

# Which programs have the strongest spatial gradient?
print("Program gradients:")
print(gradient_programs.program_gradients.sort_values('coef', key=abs, ascending=False))

# Plot the top program
top_program = gradient_programs.program_gradients['coef'].abs().idxmax()
sj.visualization.plot_gradient_curve(gradient_programs, programs=[top_program])

---
## Quick Reference

```python
import spatioloji_s.spatial.point as spoint  # same as sj.spatial.polygon

# -- Interface detection (grid-based, no graph needed) ------
result = spoint.identify_interface(sp,
    group_col='cell_type', region_a='Tumor', region_b='Stroma',
    grid_resolution=50, min_interface_cells=10)

# -- Region classification ----------------------------------
sides = spoint.classify_regions(sp, result,
    group_col='cell_type', min_segment_length=500)

# -- Gradient analysis --------------------------------------
# Gene expression
gradient = spoint.compute_gradient(sp, result,
    genes=['EPCAM', 'VIM'], layer='log_normalized')

# Cell meta features
gradient = spoint.compute_gradient(sp, result,
    features=['morph_area', 'morph_circularity'])

# Both together
gradient = spoint.compute_gradient(sp, result,
    genes=['EPCAM'], features=['morph_area'], layer='log_normalized')

# Auto-discover programs
gradient = spoint.compute_gradient(sp, result,
    auto_programs='nmf', n_auto_programs=5, layer='log_normalized')

# -- Infiltration scoring -----------------------------------
infiltration = spoint.score_infiltration(sp, result,
    immune_col='cell_type', immune_types=['CD8_T', 'Macrophage'])

# -- Visualization ------------------------------------------
sj.visualization.plot_interface_point_map(sp, result)
sj.visualization.plot_interface_point_metrics(result, metric='tortuosity')
sj.visualization.plot_gradient_curve(gradient)
sj.visualization.plot_spatial_distance(sp, gradient.distances, interface_result=result)
sj.visualization.plot_infiltration_summary(infiltration)
```

## Interpretation Guide

| Metric | Value | Meaning |
|--------|-------|---------|
| **Tortuosity** | ~1.0 | Smooth pushing border (better prognosis) |
| | >1.5 | Irregular invasive front (worse prognosis) |
| **Infiltration fraction** | High | Immune cells penetrating target region |
| | Low | Immune cells excluded from target region |
| **Gradient coef** | Positive | Expression increases toward region A |
| | Negative | Expression increases toward region B |
| | ~0 | No spatial gradient |
| **Density slope** | Negative toward target | Immune density drops with depth (excluded) |
| | Flat | Even distribution (fully infiltrated) |